# 02_modeling.ipynb — Titanic modeling and regression

This notebook reads `titanic.csv` (written by 01_eda.ipynb) and builds classification models for survival and a linear regression for fare. It performs a stratified train/test split before preprocessing, uses Pipelines with ColumnTransformer, evaluates multiple classifiers, compares resampling strategies (baseline, class_weight='balanced', SMOTE), runs GridSearchCV on RandomForest with oob_score, reports metrics, trains a linear regression for fare with MAE/RMSE/R2/Adj-R2 and saves the pipeline with joblib.


In [ ]:
# Load libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, mean_absolute_error, mean_squared_error, r2_score)
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import joblib

# Read the titanic CSV created by the EDA notebook
df = pd.read_csv('titanic.csv')
print('Loaded titanic.csv with shape', df.shape)


## Preprocessing and split (stratified) — split before any preprocessing

In [ ]:
# Select features and target for classification (exclude leakage: 'alive','class','embark_town')
features = ['pclass','sex','age','sibsp','parch','fare','embarked']
target = 'survived'
X = df[features].copy()
y = df[target].copy()

# Basic preprocessing split — stratify on target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train shape:', X_train.shape, 'Test shape:', X_test.shape)

# Column groups
numeric_features = ['age','sibsp','parch','fare']
categorical_features = ['pclass','sex','embarked']

# Preprocessing transformers fit on train only
numeric_transformer = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
categorical_transformer = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))])
preprocessor = ColumnTransformer([('num', numeric_transformer, numeric_features), ('cat', categorical_transformer, categorical_features)])

# Pipelines for classifiers
clf_pipe = Pipeline([('pre', preprocessor), ('clf', LogisticRegression(max_iter=1000))])
dt_pipe = Pipeline([('pre', preprocessor), ('clf', DecisionTreeClassifier(random_state=42))])
rf_pipe = Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(random_state=42, oob_score=True, bootstrap=True))])


## Train baseline classifiers and evaluate

In [ ]:
def eval_clf(pipeline, X_train, X_test, y_train, y_test):
    pipeline.fit(X_train, y_train)
    preds = pipeline.predict(X_test)
    probs = pipeline.predict_proba(X_test)[:,1] if hasattr(pipeline, 'predict_proba') else None
    return {
        'accuracy': accuracy_score(y_test, preds),
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall': recall_score(y_test, preds, zero_division=0),
        'f1': f1_score(y_test, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_test, probs) if probs is not None else None,
        'confusion': confusion_matrix(y_test, preds)
    }

results = {}
for name, pipe in [('LogisticRegression', clf_pipe), ('DecisionTree', dt_pipe), ('RandomForest', rf_pipe)]:
    print('Training', name)
    results[name] = eval_clf(pipe, X_train, X_test, y_train, y_test)
    print(name, results[name])


## Imbalance strategies: baseline vs class_weight='balanced' vs SMOTE (on train only)

In [ ]:
# class_weight balanced (for logistic and tree)
from copy import deepcopy
lr_bal = Pipeline([('pre', preprocessor), ('clf', LogisticRegression(class_weight='balanced', max_iter=1000))])
rf_bal = Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(class_weight='balanced', random_state=42, oob_score=True, bootstrap=True))])

# SMOTE pipeline for resampling on train only using ImbPipeline
smote_pipe = ImbPipeline([('pre', preprocessor), ('smote', SMOTE(random_state=42)), ('clf', RandomForestClassifier(random_state=42))])

imb_results = {}
for name, pipe in [('LR_balanced', lr_bal), ('RF_balanced', rf_bal), ('RF_SMOTE', smote_pipe)]:
    print('Training', name)
    imb_results[name] = eval_clf(pipe, X_train, X_test, y_train, y_test)
    print(name, imb_results[name])


## GridSearchCV on RandomForest with oob_score and reporting best params + oob_score_

In [ ]:
param_grid = {'clf__n_estimators': [50,100], 'clf__max_depth': [None, 5, 10], 'clf__max_features': ['auto','sqrt']}
rf_gs_pipe = Pipeline([('pre', preprocessor), ('clf', RandomForestClassifier(random_state=42, oob_score=True, bootstrap=True))])
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
gs = GridSearchCV(rf_gs_pipe, param_grid, cv=cv, scoring='f1', n_jobs=-1)
gs.fit(X_train, y_train)
print('Best params:', gs.best_params_)
best_rf = gs.best_estimator_
# After GridSearch, access the oob_score_ if available on the final RandomForest step
rf_step = best_rf.named_steps['clf']
if hasattr(rf_step, 'oob_score_'):
    print('OOB score reported by RF step:', rf_step.oob_score_)

# Save best RF pipeline
joblib.dump(best_rf, 'best_random_forest_pipeline.joblib')
print('Saved best_random_forest_pipeline.joblib')


## Linear regression predicting fare with residuals and metrics

In [ ]:
# Prepare a regression dataset (predict fare)
reg_features = ['pclass','sex','age','sibsp','parch','embarked']
Xr = df[reg_features].copy()
yr = df['fare'].copy()
# Drop rows where fare is missing
mask = yr.notna()
Xr = Xr[mask]
yr = yr[mask]
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)
num_reg = ['age','sibsp','parch']
cat_reg = ['pclass','sex','embarked']
pre_reg = ColumnTransformer([('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_reg), ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_reg)])
reg_pipe = Pipeline([('pre', pre_reg), ('reg', LinearRegression())])
reg_pipe.fit(Xr_train, yr_train)
preds = reg_pipe.predict(Xr_test)
mae = mean_absolute_error(yr_test, preds)
try:
    rmse = mean_squared_error(yr_test, preds, squared=False)
except TypeError:
    rmse = mean_squared_error(yr_test, preds) ** 0.5
r2 = r2_score(yr_test, preds)
n = len(yr_test); p = Xr_test.shape[1] if isinstance(Xr_test, pd.DataFrame) else reg_pipe.named_steps['pre'].transform(Xr_test).shape[1]
adj_r2 = 1 - (1-r2)*(n-1)/(n-p-1) if n - p - 1 > 0 else float('nan')
print('Regression metrics: MAE', mae, 'RMSE', rmse, 'R2', r2, 'Adj R2', adj_r2)
# Residual plot
residuals = yr_test - preds
plt.figure(figsize=(6,4))
plt.scatter(preds, residuals, alpha=0.6)
plt.axhline(0, color='red', linestyle='--')
plt.xlabel('Predicted fare')
plt.ylabel('Residuals')
plt.title('Residuals vs Predicted Fare')
plt.show()
# Save regression pipeline
joblib.dump(reg_pipe, 'fare_regression_pipeline.joblib')
print('Saved fare_regression_pipeline.joblib')


## Reload pipeline and example predict on a raw row

In [ ]:
loaded = joblib.load('best_random_forest_pipeline.joblib')
sample = X_test.head(1)
print('Sample raw row:', sample.to_dict(orient='records'))
print('Prediction from loaded pipeline:', loaded.predict(sample))
